# Structure Descent

1. **Data Preparation** — clean, feature engineering, choice sets
2. **DSL Feature Extraction** — build feature matrices
3. **Inner Loop** — fit hierarchical weights for initial structure
4. **Outer Loop** — LLM-guided structure descent (resumes from last iteration)
5. **Evaluation** — metrics, baselines, ablations



---
## 0. Setup

In [1]:
import sys
import os

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv()

from src.data_prep import load_data, clean_purchases, compute_state_features, build_choice_sets, temporal_split, join_survey, attach_train_popularity
from src.dsl import DSLStructure, DSLTerm, LAYER1_PRIMITIVES, LAYER2_AMAZON, ALL_TERMS, build_structure_features
from src.inner_loop import run_inner_loop, compute_posterior_score, fit_weights_no_hierarchy
from src.outer_loop import (
    generate_diagnostics, prompt_llm, apply_proposal, structure_descent_step,
    run_structure_descent, random_structure_search, prompt_llm_unconstrained,
)
from src.evaluation import (
    compute_metrics, compute_residuals, print_metrics,
    breakdown_by_category, breakdown_by_repeat_vs_novel,
    breakdown_by_activity_level, breakdown_by_time_of_day,
    frequency_baseline, markov_chain_baseline,
    category_switching_matrix, posterior_predictive_checks,
    simulate_sequences, no_hierarchy_weight_fn,
)
from src.subsample import subsample_customers, apply_subsample
from src.baselines.data import load_from_checkpoints, make_baseline_batch
from src.baselines.run_all import run_all_baselines, format_table
from src.checkpoint import Checkpoint

sns.set_theme(style='whitegrid')
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
ckpt = Checkpoint('data')

print('Pipeline status:')
for stage, status in ckpt.status().items():
    print(f'  {stage:15s} {status}')
print()

Pipeline status:
  data_prep       not started
  features        complete
  initial_fit     not started
  outer_loop      not started
  final           not started



---
## 1. Data Preparation

In [2]:
if ckpt.has('data_prep'):
    print('Found data_prep checkpoint — loading...')
    df = ckpt.load_file('purchases_processed.parquet')
    train_choices = ckpt.load_file('train_choices.pkl')
    val_choices = ckpt.load_file('val_choices.pkl')
    test_choices = ckpt.load_file('test_choices.pkl')
    # New format: `event_weights.pkl` is an np.ndarray aligned with train_choices
    # (one weight per training event). Older checkpoints may still have the
    # legacy `event_weight_map.pkl`; we handle both.
    if (DATA_DIR / 'event_weights.pkl').exists():
        event_weights_array = ckpt.load_file('event_weights.pkl')
    elif (DATA_DIR / 'event_weight_map.pkl').exists():
        # Legacy dict — convert to array via train_choices customer_id lookup.
        _legacy = ckpt.load_file('event_weight_map.pkl')
        event_weights_array = np.asarray(
            [float(_legacy.get(ev['customer_id'], 1.0)) for ev in train_choices],
            dtype=float,
        )
    else:
        event_weights_array = None
    print(f'  {len(df):,} purchases, {len(train_choices):,} train / {len(val_choices):,} val / {len(test_choices):,} test')
else:
    print('Running data preparation...')
    purchases_raw, survey = load_data()
    df = clean_purchases(purchases_raw)
    df = join_survey(df, survey)
    df = compute_state_features(df)

    MIN_PURCHASES = 5
    counts = df['customer_id'].value_counts()
    df = df[df['customer_id'].isin(counts[counts >= MIN_PURCHASES].index)].reset_index(drop=True)
    df = temporal_split(df, val_frac=0.1, test_frac=0.1)
    df = attach_train_popularity(df)

    SUBSAMPLE_CUSTOMERS = 500  # increased from 100 for better term acceptance
    event_weights_array = None
    if SUBSAMPLE_CUSTOMERS is not None:
        train_df_full = df[df['split'] == 'train'].reset_index(drop=True)
        selected_ids, customer_weights = subsample_customers(train_df_full, n_customers=SUBSAMPLE_CUSTOMERS, seed=42)
        # apply_subsample now returns (filtered_df, event_weights) where
        # event_weights is an np.ndarray of length len(filtered_df) aligned
        # with filtered_df's reset-index row order. Because build_choice_sets
        # iterates train_df_sub in iloc order without re-sorting, the array
        # stays aligned with train_choices.
        train_df_sub, event_weights_array = apply_subsample(train_df_full, selected_ids, customer_weights)
        print(f'Subsampled: {len(selected_ids)} customers, {len(train_df_sub):,} events')
    else:
        train_df_sub = None

    train_df = train_df_sub if train_df_sub is not None else df[df['split'] == 'train'].reset_index(drop=True)
    train_choices = build_choice_sets(train_df, n_negatives=9)
    val_choices = build_choice_sets(df[df['split'] == 'val'].reset_index(drop=True), n_negatives=9)
    test_choices = build_choice_sets(df[df['split'] == 'test'].reset_index(drop=True), n_negatives=9)

    # Sanity check: event_weights_array length must match train_choices length.
    if event_weights_array is not None:
        assert len(event_weights_array) == len(train_choices), (
            f'event_weights_array length {len(event_weights_array)} != '
            f'train_choices length {len(train_choices)} — alignment broken'
        )

    save_data = {
        'purchases_processed.parquet': df,
        'train_choices.pkl': train_choices,
        'val_choices.pkl': val_choices,
        'test_choices.pkl': test_choices,
    }
    if event_weights_array is not None:
        save_data['event_weights.pkl'] = event_weights_array
    ckpt.save('data_prep', save_data, list(save_data.keys()))
    print(f'  {len(train_choices):,} train / {len(val_choices):,} val / {len(test_choices):,} test')

Running data preparation...
[load_data] Loading purchases from /Users/wesleylu/Desktop/structure_descent/amazon_ecom/amazon-purchases.csv ...
[load_data] Loaded 1,850,717 rows, 8 columns
[load_data] Loading survey from /Users/wesleylu/Desktop/structure_descent/amazon_ecom/survey.csv ...
[load_data] Loaded 5,027 survey respondents
[clean] Starting with 1,850,717 rows ...
[clean] Dropped 973 rows with missing customer_id/order_date/asin
[clean] Done: 1,849,744 rows, 5,023 customers, 939,082 ASINs, 1,869 categories
[survey] Joining survey onto 1,849,744 events ...
[survey] Done: added 22 survey columns (0 all-null dropped)
[features] Computing state features ...
[features]   routine (repeat purchase count) ...
[features]   recency_days (days since last purchase of item) ...
[features]   cat_affinity (category purchase count) ...
[features]   cat_count_7d / cat_count_30d (time-based rolling counts of events) ...
[features]   brand (mode first-token per ASIN) ...
[features] Done: 1,849,744 

/Users/wesleylu/Desktop/structure_descent/src/subsample.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  repeat_rate = grouped.apply(lambda g: (g["routine"] > 0).mean()).rename("repeat_rate")
/Users/wesleylu/Desktop/structure_descent/src/subsample.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mean_recency = grouped.apply(
/Users/wesleylu/Desktop/structure_descent/src/subsample.py:38: FutureWarning: DataFra

Subsampled: 500 customers, 119,948 events
[choice_sets] Building from 119,948 events (n_resamples=1) ...
[choice_sets] 81,981 unique ASINs, 1,584 categories, 9 negatives each
[choice_sets] Done: 119,948 choice sets built
[choice_sets] Building from 182,771 events (n_resamples=1) ...
[choice_sets] 133,845 unique ASINs, 1,664 categories, 9 negatives each
[choice_sets] Done: 182,771 choice sets built
[choice_sets] Building from 182,771 events (n_resamples=1) ...
[choice_sets] 133,740 unique ASINs, 1,657 categories, 9 negatives each
[choice_sets] Done: 182,771 choice sets built
  [Checkpoint] Saved: data_prep (5 files)
  119,948 train / 182,771 val / 182,771 test


In [3]:
# Leakage fix (2026-04-15): feature-extraction semantics changed, so any
# data_prep/features checkpoints produced before the fix are invalid (they
# contain lookahead-leaked positive rows). Clear them here so a rerun of the
# earlier cells regenerates them under the fixed code.
ckpt.clear('data_prep')
ckpt.clear('features')
ckpt.clear('initial_fit')
ckpt.clear('outer_loop')
print('Clear. Ready to rerun')

Clear. Ready to rerun


---
## 2. DSL Feature Extraction

In [4]:
if ckpt.has('features'):
    print('Found features checkpoint — loading...')
    feat_data = ckpt.load_file('train_features.pkl')
    full_features_list = feat_data['features_list']
    chosen_indices = feat_data['chosen_indices']
    customer_ids = feat_data['customer_ids']
    categories = feat_data['categories']
    all_term_names = feat_data['all_term_names']
    structure = DSLStructure.from_dict(feat_data['structure'])
    print(f'  {len(full_features_list):,} events, {len(all_term_names)} terms: {all_term_names}')
else:
    print('Building leakage-safe lookup tables...')
    # TRAIN-ONLY popularity. The `popularity` column attached by
    # attach_train_popularity is already train-derived (val/test rows are
    # broadcast from train counts; ASINs unseen in train get 0). We just need
    # to build an asin -> log1p(train_count) map. Do NOT use df.groupby('asin')
    # because that would re-count val/test rows.
    _train_mask = df['split'] == 'train'
    _train_asin_counts = df.loc[_train_mask].groupby('asin').size()
    item_popularity_train = np.log1p(_train_asin_counts).to_dict()

    # Category average price — this is a catalog-level statistic used only as a
    # normalization scale for price_sensitivity/price_rank. It's not a
    # per-event feature, so using the whole df for stability is fine (train/
    # val/test rows share the same catalog). If you want strict train-only,
    # swap the source to df.loc[_train_mask].
    cat_avg_price = df.groupby('category')['price'].mean().to_dict()
    asin_category = df.groupby('asin')['category'].first().to_dict()

    # All 12 DSL terms — extract once, slice later
    all_term_names = ALL_TERMS  # ['routine', 'recency', ..., 'co_purchase']

    def _chosen_feature_row(event):
        """Feature row for the positive alternative, from leakage-safe
        chosen_features attached at choice-set construction time."""
        cf = event['chosen_features']
        routine_val = float(cf.get('routine', 0.0))
        recency_val = 1.0 / (1.0 + float(cf.get('recency_days', 999.0)))
        novelty_val = float(cf.get('novelty', 1.0))
        aff_val = float(np.log1p(float(cf.get('cat_affinity', 0.0))))
        pop_val = float(cf.get('popularity', 0.0))
        # pop_val is a train-only integer count; log-compress for scale parity
        pop_val = float(np.log1p(pop_val))
        item_price = float(cf.get('price', 0.0) or 0.0)
        event_category = event['category']
        avg_price = cat_avg_price.get(event_category, 10.0)
        price_sens = -(item_price / (avg_price + 1e-8) - 1.0)
        price_rank_val = 1.0 - (item_price / (avg_price + 1e-8))
        # brand_affinity requires per-event history; we don't have a
        # leakage-safe per-event brand-affinity column, so set to 0 for both
        # chosen and negatives (symmetric, no leakage). Models that care about
        # brand can learn it via the brand id elsewhere.
        return {
            'routine': routine_val, 'recency': recency_val, 'novelty': novelty_val,
            'popularity': pop_val, 'affinity': aff_val, 'time_match': 0.0,
            'price_sensitivity': price_sens, 'rating_signal': 0.0,
            'brand_affinity': 0.0, 'price_rank': price_rank_val,
            'delivery_speed': 0.0, 'co_purchase': 0.0,
        }

    def _negative_feature_row(asin, event_category):
        """Feature row for a negative alternative. No per-customer history is
        available (customer never touched this item by construction), so all
        history-derived features take their 'never-seen' defaults.
        Popularity is train-only; price features use the category average
        (we don't know the negative item's price without leaking)."""
        pop_val = float(item_popularity_train.get(asin, 0.0))
        return {
            'routine': 0.0,
            'recency': 1.0 / (1.0 + 999.0),
            'novelty': 1.0,
            'popularity': pop_val,
            'affinity': 0.0,
            'time_match': 0.0,
            'price_sensitivity': 0.0,
            'rating_signal': 0.0,
            'brand_affinity': 0.0,
            'price_rank': 0.0,
            'delivery_speed': 0.0,
            'co_purchase': 0.0,
        }

    def build_full_feature_matrix(event):
        """Build [n_alts x ALL_TERMS] feature matrix — all 12 terms.

        The positive alternative is populated from event['chosen_features']
        (the customer's causal state at the event's timestamp). Negative
        alternatives use the symmetric 'never-seen' defaults plus train-only
        item popularity. This avoids the leakage bug where .groupby().last()
        would leak future customer-item state into the positive row.
        """
        n_alt = len(event['choice_asins'])
        mat = np.zeros((n_alt, len(all_term_names)))
        chosen_idx = event['chosen_idx']
        cat = event['category']
        chosen_feat = _chosen_feature_row(event)
        for k, asin in enumerate(event['choice_asins']):
            if k == chosen_idx:
                feat = chosen_feat
            else:
                feat = _negative_feature_row(asin, cat)
            for j, term in enumerate(all_term_names):
                mat[k, j] = feat.get(term, 0.0)
        return mat

    structure = DSLStructure.initial()
    MAX_EVENTS = None  # set to 10_000 for a quick test
    events = train_choices[:MAX_EVENTS] if MAX_EVENTS else train_choices

    full_features_list, chosen_indices, customer_ids, categories = [], [], [], []
    for event in tqdm(events, desc='Extracting features (all 12 terms)'):
        full_features_list.append(build_full_feature_matrix(event))
        chosen_indices.append(event['chosen_idx'])
        customer_ids.append(event['customer_id'])
        categories.append(event['category'])

    ckpt.save('features', {
        'train_features.pkl': {
            'features_list': full_features_list, 'chosen_indices': chosen_indices,
            'customer_ids': customer_ids, 'categories': categories,
            'structure': structure.to_dict(),
            'all_term_names': all_term_names,
        }
    }, ['train_features.pkl'])
    print(f'  {len(full_features_list):,} events, shape: {full_features_list[0].shape} ({len(all_term_names)} terms)')

# ── Helper: build features for any structure (simple + compound terms) ──
from src.dsl import build_structure_features

def get_features_for_structure(s):
    """Build feature matrices for a structure, supporting compound terms."""
    return [build_structure_features(s, f, all_term_names) for f in full_features_list]

# Build initial features_list for the starting structure
features_list = get_features_for_structure(structure)
print(f'Initial structure features: {features_list[0].shape}')

Building leakage-safe lookup tables...


Extracting features (all 12 terms):   0%|          | 0/119948 [00:00<?, ?it/s]

  [Checkpoint] Saved: features (1 files)
  119,948 events, shape: (10, 12) (12 terms)
Initial structure features: (10, 2)


In [5]:
# ── Save raw 12-column base feature matrices for the new baseline suite ──
#
# The legacy `train_features.pkl` above holds the per-structure projection and
# uses an older schema. The new baseline suite (src/baselines/) expects
# `{train,val,test}_base_features.pkl` with schema {features_list, feature_names}
# — one raw 12-column matrix per event, per split. This cell computes and
# saves those files so `src.baselines.data.load_from_checkpoints("data")`
# works end-to-end. Safe to re-run: writes overwrite.
#
# LEAKAGE-SAFE: positive-alt features come from event['chosen_features']
# (the customer's causal state at the event's timestamp, set in
# build_choice_sets). Negative alts use symmetric 'never-seen' defaults +
# train-only popularity. Val/test events use the same scheme — their
# chosen_features were also captured at the event's timestamp.

base_needed = ['train_base_features.pkl', 'val_base_features.pkl', 'test_base_features.pkl']
all_present = all((DATA_DIR / f).exists() for f in base_needed)

if all_present:
    print('All base_features files already present — skipping recomputation.')
else:
    print('Computing raw 12-column base features for train/val/test...')

    # Train-only popularity. See cell 7 for the rationale.
    _train_mask = df['split'] == 'train'
    _train_asin_counts = df.loc[_train_mask].groupby('asin').size()
    item_popularity_train = np.log1p(_train_asin_counts).to_dict()
    cat_avg_price = df.groupby('category')['price'].mean().to_dict()

    def _chosen_feature_row_v2(event):
        cf = event['chosen_features']
        routine_v  = float(cf.get('routine', 0.0))
        recency_v  = 1.0 / (1.0 + float(cf.get('recency_days', 999.0)))
        novelty_v  = float(cf.get('novelty', 1.0))
        aff_v      = float(np.log1p(float(cf.get('cat_affinity', 0.0))))
        pop_v      = float(np.log1p(float(cf.get('popularity', 0.0))))
        item_price = float(cf.get('price', 0.0) or 0.0)
        avg_price  = cat_avg_price.get(event['category'], 10.0)
        price_s    = -(item_price / (avg_price + 1e-8) - 1.0)
        price_r    = 1.0 - (item_price / (avg_price + 1e-8))
        return {
            'routine': routine_v, 'recency': recency_v, 'novelty': novelty_v,
            'popularity': pop_v, 'affinity': aff_v, 'time_match': 0.0,
            'price_sensitivity': price_s, 'rating_signal': 0.0,
            'brand_affinity': 0.0, 'price_rank': price_r,
            'delivery_speed': 0.0, 'co_purchase': 0.0,
        }

    def _negative_feature_row_v2(asin):
        pop_v = float(item_popularity_train.get(asin, 0.0))
        return {
            'routine': 0.0,
            'recency': 1.0 / (1.0 + 999.0),
            'novelty': 1.0,
            'popularity': pop_v,
            'affinity': 0.0,
            'time_match': 0.0,
            'price_sensitivity': 0.0,
            'rating_signal': 0.0,
            'brand_affinity': 0.0,
            'price_rank': 0.0,
            'delivery_speed': 0.0,
            'co_purchase': 0.0,
        }

    def _event_feature_matrix(event, term_names):
        n_alt = len(event['choice_asins'])
        mat   = np.zeros((n_alt, len(term_names)), dtype=float)
        chosen_idx = event['chosen_idx']
        chosen_feat = _chosen_feature_row_v2(event)
        for k, asin in enumerate(event['choice_asins']):
            f = chosen_feat if k == chosen_idx else _negative_feature_row_v2(asin)
            for j, term in enumerate(term_names):
                mat[k, j] = f.get(term, 0.0)
        return mat

    base_term_names = list(ALL_TERMS)
    split_to_choices = {
        'train': train_choices,
        'val':   val_choices,
        'test':  test_choices,
    }

    for split_name, split_choices in split_to_choices.items():
        features_list = [
            _event_feature_matrix(ev, base_term_names)
            for ev in tqdm(split_choices, desc=f'{split_name} base features')
        ]
        out_path = DATA_DIR / f'{split_name}_base_features.pkl'
        with open(out_path, 'wb') as f:
            pickle.dump(
                {'features_list': features_list, 'feature_names': base_term_names},
                f,
            )
        print(f'  Saved {len(features_list):,} {split_name} events -> {out_path.name}')

    print('Done. src.baselines.data.load_from_checkpoints("data") will now work.')


All base_features files already present — skipping recomputation.


---
## 3. Inner Loop: Fit Initial Weights

In [6]:
metadata = [e['metadata'] for e in train_choices[:len(full_features_list)]]

# Slice event_weights_array to match the (possibly truncated) feature list.
# By construction event_weights_array[i] corresponds to train_choices[i], which
# in turn corresponds to full_features_list[i], so a prefix slice is aligned.
event_weights_fit = (
    np.asarray(event_weights_array, dtype=float)[:len(full_features_list)]
    if event_weights_array is not None
    else None
)

if ckpt.has('initial_fit'):
    print('Found initial_fit checkpoint — loading...')
    state = ckpt.load_file('current_state.pkl')
    structure = DSLStructure.from_dict(state['structure'])
    score = state['score']
    metrics = state['metrics']
    residuals = state['residuals']
    features_list = get_features_for_structure(structure)
    weights, _ = run_inner_loop(structure, features_list, chosen_indices, customer_ids, categories, sigma=10.0, tau=1.0, nu=0.5, event_weights=event_weights_fit)
    print(f'  Structure: {structure}  |  Score: {score:.2f}')
else:
    print('Fitting initial weights...')
    features_list = get_features_for_structure(structure)
    weights, score = run_inner_loop(structure, features_list, chosen_indices, customer_ids, categories, sigma=10.0, tau=1.0, nu=0.5, event_weights=event_weights_fit)
    metrics = compute_metrics(features_list, chosen_indices, weights.get_weights, customer_ids, categories)
    residuals = compute_residuals(features_list, chosen_indices, weights.get_weights, customer_ids, categories, metadata)

    print(f'Posterior score: {score:.2f}')
    print('Global weights (theta_g):')
    for term, w in zip(structure.terms, weights.theta_g):
        print(f'  {term.display_name:25s}  {w:+.4f}')

    ckpt.save('initial_fit', {
        'current_state.pkl': {'structure': structure.to_dict(), 'score': score, 'metrics': metrics, 'residuals': residuals}
    }, ['current_state.pkl'])

Fitting initial weights...
Posterior score: -1735091.27
Global weights (theta_g):
  routine                    +0.7906
  affinity                   +2.0970
  [Checkpoint] Saved: initial_fit (1 files)


---
## 4. Outer Loop: LLM-Guided Structure Descent

Resumes from last completed iteration if interrupted.

In [ ]:
N_ITERATIONS = 15

def fit_fn(s):
    """Slice features to match proposed structure, then fit."""
    fl = get_features_for_structure(s)
    return run_inner_loop(s, fl, chosen_indices, customer_ids, categories, sigma=10.0, tau=1.0, nu=0.5, event_weights=event_weights_fit)

def get_metrics_fn(w, s):
    fl = get_features_for_structure(s)
    return compute_metrics(fl, chosen_indices, w.get_weights, customer_ids, categories)

def get_residuals_fn(w, s):
    fl = get_features_for_structure(s)
    return compute_residuals(fl, chosen_indices, w.get_weights, customer_ids, categories, metadata)

history = run_structure_descent(
    initial_structure=structure,
    fit_fn=fit_fn,
    get_metrics_fn=get_metrics_fn,
    get_residuals_fn=get_residuals_fn,
    n_iterations=N_ITERATIONS,
    checkpoint=ckpt,
)

from src.dsl import DSLTerm
_terms = history[-1]['structure'].replace('S = ', '').split(' + ')
current_structure = DSLStructure([DSLTerm.parse(t) for t in _terms])
current_score = history[-1]['score']
print(f'\nFinal structure: {current_structure}')
print(f'Final score: {current_score:.2f}')

[Strategy] greedy: {'name': 'greedy'}
Initial: S = routine + affinity  |  Score: -1735091.27
  [Checkpoint] Outer loop iter 0 saved

[Outer Loop iter=1] Querying LLM for structural proposal...
  [LLM] provider=anthropic  model=claude-opus-4-6
  [LLM] Reasoning: The model has perfect top-1 accuracy but poor calibration (NLL=0.8953), meaning the softmax probabilities are insufficiently peaked. Adding both popularity and recency provides two orthogonal signals: popularity gives a global frequency baseline that helps discriminate chosen items from random negat
  [DSL] Parsed term: popularity
  [DSL] Parsed term: recency
  [Proposal] S = routine + affinity  →  S = routine + affinity + popularity + recency
  [Inner Loop] Fitting weights for proposed structure...
  [Reject/greedy] Proposed score -2141789.70 vs current -1735091.27
  [Checkpoint] Outer loop iter 1 saved
  [Viz] Exported search history to data/search_history.json
  [History] 1 prior proposals (0 accepted, 1 rejected)
    ✗ iter 

In [ ]:
# ── Search history plot ──
hist_df = pd.DataFrame(history)
display(hist_df)

plt.figure(figsize=(8, 3))
plt.plot(hist_df['iteration'], hist_df['score'], marker='o')
plt.scatter(hist_df[hist_df['accepted']]['iteration'],
            hist_df[hist_df['accepted']]['score'], color='green', zorder=5, label='Accepted')
plt.xlabel('Outer loop iteration')
plt.ylabel('Posterior score')
plt.title('Structure Descent: posterior score over iterations')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Save final structure ──
with open(DATA_DIR / 'final_structure.pkl', 'wb') as f:
    pickle.dump({'structure': current_structure.to_dict(),
                 'score': current_score, 'history': history}, f)
ckpt.save('final', {}, ['final_structure.pkl'])
print(f'Saved. Final: {current_structure}  |  Score: {current_score:.2f}')

---
## 5. Evaluation

In [ ]:
final_structure = current_structure
features_list = get_features_for_structure(final_structure)
weights, score = run_inner_loop(
    final_structure, features_list, chosen_indices, customer_ids, categories
)
metrics = compute_metrics(
    features_list, chosen_indices, weights.get_weights, customer_ids, categories
)
print_metrics(metrics, label='Structure Descent (final)')
print(f'Posterior score: {score:.2f}')

### A. Metric Breakdowns

In [ ]:
# ── By category ──
cat_breakdown = breakdown_by_category(
    features_list, chosen_indices, weights.get_weights, customer_ids, categories
)
top20 = cat_breakdown.nlargest(20, 'n_events')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top20['category'], top20['top1'], color='steelblue')
ax.axvline(metrics['top1'], color='red', linestyle='--', label=f'Overall ({metrics["top1"]:.1%})')
ax.set_xlabel('Top-1 Accuracy')
ax.set_title('Top-1 by Category (top 20 by volume)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Repeat vs. novel ──
repeat_breakdown = breakdown_by_repeat_vs_novel(
    features_list, chosen_indices, weights.get_weights,
    customer_ids, categories, metadata
)
display(repeat_breakdown)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(repeat_breakdown['type'], repeat_breakdown['top1'], color=['steelblue', 'coral'])
ax.set_ylabel('Top-1 Accuracy')
ax.set_title('Accuracy: Repeat vs. Novel Purchases')
plt.tight_layout()
plt.show()

In [ ]:
# ── By activity level ──
purchase_counts = df['customer_id'].value_counts().to_dict()
activity_breakdown = breakdown_by_activity_level(
    features_list, chosen_indices, weights.get_weights,
    customer_ids, categories, purchase_counts, n_bins=3
)
display(activity_breakdown)

### B. Baselines

In [ ]:
# ── Frequency baseline ──
cust_history = defaultdict(lambda: defaultdict(int))
for ev in train_choices[:len(features_list)]:
    cust_history[ev['customer_id']][ev['chosen_asin']] += 1

freq_metrics = frequency_baseline(
    train_choices[:len(features_list)],
    {cid: dict(counts) for cid, counts in cust_history.items()}
)
print_metrics(freq_metrics, label='Frequency baseline')

In [ ]:
# ── Markov chain baseline ──
trans_matrix = category_switching_matrix(df, top_n=50)
cat_popularity = defaultdict(lambda: defaultdict(int))
for ev in train_choices[:len(features_list)]:
    cat_popularity[ev['category']][ev['chosen_asin']] += 1

prev_cat_map = {}
events_with_prev = []
for ev in train_choices[:len(features_list)]:
    ev_copy = dict(ev)
    ev_copy['prev_category'] = prev_cat_map.get(ev['customer_id'], ev['category'])
    prev_cat_map[ev['customer_id']] = ev['category']
    events_with_prev.append(ev_copy)

markov_metrics = markov_chain_baseline(
    events_with_prev, trans_matrix,
    {cat: dict(counts) for cat, counts in cat_popularity.items()}
)
print_metrics(markov_metrics, label='Markov chain baseline')

In [ ]:
# ── Hand-specified logit (all terms, no search) ──
hand_structure = DSLStructure(LAYER1_PRIMITIVES + LAYER2_AMAZON)
hand_features = get_features_for_structure(hand_structure)

hand_weights, hand_score = run_inner_loop(
    hand_structure, hand_features, chosen_indices, customer_ids, categories
)
hand_metrics = compute_metrics(
    hand_features, chosen_indices, hand_weights.get_weights, customer_ids, categories
)
print_metrics(hand_metrics, label='Hand-specified logit')

### C. Ablations

In [ ]:
# ── No hierarchy ablation ──
theta_g_only = fit_weights_no_hierarchy(final_structure, features_list, chosen_indices, sigma=10.0)
flat_weight_fn = no_hierarchy_weight_fn(theta_g_only)
flat_metrics = compute_metrics(
    features_list, chosen_indices, flat_weight_fn, customer_ids, categories
)
print_metrics(flat_metrics, label='No hierarchy (theta_g only)')
print_metrics(metrics, label='Full hierarchy')

In [ ]:
# ── Random search ablation ──
def fit_fn_ablation(s):
    fl = get_features_for_structure(s)
    return run_inner_loop(s, fl, chosen_indices, customer_ids, categories)

random_history = random_structure_search(DSLStructure.initial(), fit_fn_ablation, n_iterations=8, seed=42)

# ── Compare LLM vs random ──
llm_hist_df = pd.DataFrame(history)
random_hist_df = pd.DataFrame(random_history)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(llm_hist_df['iteration'], llm_hist_df['score'], marker='o', label='LLM-guided', color='steelblue')
ax.plot(random_hist_df['iteration'], random_hist_df['score'], marker='s', label='Random search', color='coral', linestyle='--')
ax.set_xlabel('Outer loop iteration')
ax.set_ylabel('Posterior score')
ax.set_title('Structure Descent: LLM-guided vs. Random DSL Search')
ax.legend()
plt.tight_layout()
plt.show()

### D. Full Comparison

In [ ]:
def fmt(m, key):
    v = m.get(key, float('nan'))
    if key in ('top1', 'top5'):
        return f'{v:.1%}' if not np.isnan(v) else 'n/a'
    return f'{v:.4f}' if not np.isnan(v) else 'n/a'

rows = [
    ('Frequency', freq_metrics),
    ('Markov chain', markov_metrics),
    ('Hand-specified logit', hand_metrics),
    ('No hierarchy (ablation)', flat_metrics),
    ('Structure Descent (LLM)', metrics),
]

comparison = pd.DataFrame([
    {'Model': name, 'Top-1': fmt(m,'top1'), 'Top-5': fmt(m,'top5'),
     'MRR': fmt(m,'mrr'), 'NLL': fmt(m,'val_nll')}
    for name, m in rows
])
display(comparison)

### E. Interpretability — Final Structure

In [ ]:
print(f'Final structure: {final_structure}')
print(f'Complexity L(S) = {final_structure.complexity()}')
print(f'log p(S) = {final_structure.log_prior():.4f}')
print(f'Posterior score = {score:.2f}')

print('\nGlobal weights (sorted by magnitude):')
for term, w in sorted(zip(final_structure.terms, weights.theta_g), key=lambda x: -abs(x[1])):
    print(f'  {term.display_name:30s}  {w:+.4f}')

sorted_pairs = sorted(zip(final_structure.terms, weights.theta_g), key=lambda x: x[1])
terms_s, weights_s = zip(*sorted_pairs)

fig, ax = plt.subplots(figsize=(7, max(3, len(terms_s) * 0.45)))
colors = ['steelblue' if w >= 0 else 'tomato' for w in weights_s]
ax.barh(terms_s, weights_s, color=colors)
ax.axvline(0, color='k', linewidth=0.8)
ax.set_xlabel('Global weight (theta_g)')
ax.set_title('Structure Descent — Final Discovered Structure')
plt.tight_layout()
plt.show()

---
## F. New Baseline Suite — 8 Methods

This section runs the full 8-baseline suite against the same train/val/test splits produced above. Each baseline is fit on train (with val for hyperparameter selection where applicable) and scored on test through the shared harness at `src/baselines/evaluate.py`.

The baselines in the registry are:

1. **LASSO-MNL** — L1-regularized conditional logit (FISTA + soft-threshold)
2. **RandomForest** — bagging of trees
3. **GradientBoosting** — histogram-based boosting
4. **MLP** — small feed-forward neural net
5. **Paz-VNS** — Variable Neighborhood Search over DSL structures
6. **DUET-GA** — DUET ANN (linear + tanh MLP with soft monotonicity, NOT a GA despite the filename)
7. **Bayesian-ARD** — Per-coefficient ARD prior + NUTS sampler
8. **Delphos** — DQN over DSL-specification actions

Budgets below are tuned for a **first-pass run** — small enough to complete in a reasonable wall clock, large enough to produce meaningful numbers. Scale them up for a headline paper run.


In [ ]:
# ── Load train/val/test as BaselineEventBatch instances ──
from src.baselines.data import load_from_checkpoints

train_batch, val_batch, test_batch = load_from_checkpoints('data')
print(f'Loaded baseline batches:')
print(f'  train: {train_batch.n_events:,} events, {train_batch.n_alternatives} alts, {train_batch.n_base_terms} base features')
print(f'  val:   {val_batch.n_events:,} events')
print(f'  test:  {test_batch.n_events:,} events')


In [ ]:
# ── Per-baseline budgets tuned for a first-pass real-data run ──
# Scale these up for a headline paper run. See src/baselines/ per-module docs
# for full hyperparameter descriptions.

BASELINE_KWARGS = {
    # Shrinkage baselines — fast, use defaults
    'LASSO-MNL':        {},
    'Bayesian-ARD':     {'n_warmup': 300, 'n_samples': 300, 'inference': 'nuts'},

    # Classical ML ceilings — fast, use defaults
    'RandomForest':     {'n_estimators': 100, 'max_depth': 10},
    'GradientBoosting': {'max_iter': 150, 'max_depth': 6, 'learning_rate': 0.08},
    'MLP':              {'hidden_layer_sizes': (32, 16), 'max_iter': 300},

    # Search baselines — slow, bounded budgets for first-pass
    'Paz-VNS':          {'k_max': 3, 'max_evaluations': 40},
    'DUET-GA':          {'hidden': (32, 32), 'n_epochs': 80, 'patience': 15},
    'Delphos':          {'num_episodes': 80, 'early_stop_window': 20, 'patience': 10, 'min_percentage': 0.3},
}

rows = run_all_baselines(
    train_batch,
    val_batch,
    test_batch,
    baseline_kwargs=BASELINE_KWARGS,
    verbose=True,
)

# Persist raw rows for later analysis
import pickle
with open(DATA_DIR / 'baseline_suite_rows.pkl', 'wb') as f:
    pickle.dump(rows, f)
print(f'\nSaved {len(rows)} baseline result rows -> data/baseline_suite_rows.pkl')


In [ ]:
# ── Comparison table ──
print()
print(format_table(rows))
print()

# Summarize as a pandas DataFrame for notebook display
baseline_df = pd.DataFrame([
    {
        'name':        r['name'],
        'status':      r['status'],
        'top1':        r.get('top1'),
        'top5':        r.get('top5'),
        'mrr':         r.get('mrr'),
        'test_nll':    r.get('test_nll'),
        'aic':         r.get('aic'),
        'bic':         r.get('bic'),
        'n_params':    r.get('n_params'),
        'fit_seconds': r.get('fit_seconds'),
    }
    for r in rows
])
display(baseline_df.round(4))


In [ ]:
# ── Put Structure Descent alongside the new baselines ──
# The final Structure Descent fit from Section E is already on `final_structure`
# and `weights`. Compute the same metric panel on `test_batch` so it can sit
# next to the baseline rows produced above.

from src.baselines.evaluate import evaluate_baseline
from src.baselines.base import BaselineEventBatch
from src.dsl import build_structure_features

class _SDAdapter:
    name = 'Structure-Descent'
    def __init__(self, structure, weights, train_n_events):
        self.structure = structure
        self.weights = weights
        self._train_n_events = train_n_events
    def score_events(self, batch):
        out = []
        for base in batch.base_features_list:
            feats = build_structure_features(self.structure, base, batch.base_feature_names)
            # Per-event scores; customer/category info lives per-event
            out.append(feats @ self.weights.theta_g)  # global weights; no per-event hier here
        return out
    @property
    def n_params(self):
        return len(self.structure.terms) * (1 + len(self.weights.theta_c) + len(self.weights.delta_i))
    @property
    def description(self):
        return f'Structure Descent: {self.structure} (global weights only for scoring)'

sd_fitted = _SDAdapter(final_structure, weights, train_batch.n_events)
sd_report = evaluate_baseline(sd_fitted, test_batch, train_n_events=train_batch.n_events)

sd_row = {
    'name':        'Structure-Descent',
    'status':      'ok',
    'top1':        sd_report.metrics['top1'],
    'top5':        sd_report.metrics['top5'],
    'mrr':         sd_report.metrics['mrr'],
    'test_nll':    sd_report.metrics['test_nll'],
    'aic':         sd_report.metrics['aic'],
    'bic':         sd_report.metrics['bic'],
    'n_params':    sd_report.n_params,
    'fit_seconds': 0.0,  # already-fit
}
full_df = pd.concat([baseline_df, pd.DataFrame([sd_row])], ignore_index=True)
print('\n=== Full comparison: 8 baselines + Structure Descent ===')
display(full_df.round(4).sort_values('test_nll'))

with open(DATA_DIR / 'baseline_suite_full_table.pkl', 'wb') as f:
    pickle.dump(full_df, f)
print(f'\nSaved full comparison table -> data/baseline_suite_full_table.pkl')
